# 02 - SQL analysis

In [1]:
import pandas as pd
from sqlalchemy import create_engine

engine = create_engine("sqlite:///data/movies.db")

In [2]:
pd.set_option("display.max_rows", None)

In [3]:
pd.reset_option("display.max_rows")

## Action movies released after 2015 

In [4]:
q1 = """SELECT m.title, m.release_date FROM movies m
JOIN movie_genres mg ON m.movie_id = mg.movie_id
JOIN genres g ON mg.genre_id = g.genre_id 
WHERE g.genre_name='Action' AND m.release_date >= "2016-01-01"
"""

pd.read_sql(q1,engine)

,title,release_date
0,Bad Boys for Life,2020-01-15 00:00:00.000000
1,Ghostbusters,2016-07-14 00:00:00.000000
2,Independence Day: Resurgence,2016-06-22 00:00:00.000000
3,xXx: Return of Xander Cage,2017-01-13 00:00:00.000000
4,Warcraft,2016-05-25 00:00:00.000000
...,...,...
263,Superman,2025-07-09 00:00:00.000000
264,How to Train Your Dragon,2025-06-06 00:00:00.000000
265,Sinners,2025-04-16 00:00:00.000000
266,Jurassic World Rebirth,2025-06-23 00:00:00.000000


### Insight

1. 268 Action movies were released from 1 January 2016 onwards, so Action is a regularly produced genre in recent years.

2. Note: release_date is stored with a time part (for example 2016-09-06 00:00:00.000000), so the filter release_date >= '2016-01-01' is used instead of > '2015-12-31' to avoid including 31 December 2015.

Recommendation: This query only lists titles and dates, so it does not show which films performed well. To find successful Action films, this list should be combined with revenue data.



## Top 10 highest-grossing movies with their genres

In [5]:
q2 = """SELECT m.title, m.revenue, GROUP_CONCAT(g.genre_name,", ") AS genres FROM movies m
JOIN movie_genres mg ON m.movie_id = mg.movie_id
JOIN genres g ON mg.genre_id = g.genre_id
GROUP BY m.movie_id, m.title, m.revenue
ORDER BY m.revenue DESC
LIMIT 10
"""

print(pd.read_sql(q2,engine))

                          title       revenue  \
0                        Avatar  2.923706e+09   
1             Avengers: Endgame  2.799439e+09   
2      Avatar: The Way of Water  2.334485e+09   
3                       Titanic  2.264162e+09   
4  Star Wars: The Force Awakens  2.068224e+09   
5        Avengers: Infinity War  2.052415e+09   
6       Spider-Man: No Way Home  1.921207e+09   
7                    Zootopia 2  1.868209e+09   
8                  Inside Out 2  1.698864e+09   
9                Jurassic World  1.671537e+09   

                                          genres  
0             Adventure, Action, Science Fiction  
1             Adventure, Action, Science Fiction  
2             Adventure, Action, Science Fiction  
3                                 Drama, Romance  
4             Adventure, Action, Science Fiction  
5             Adventure, Action, Science Fiction  
6             Adventure, Action, Science Fiction  
7  Adventure, Animation, Comedy, Mystery, Family  
8

### Insight

1. Avatar is the highest-grossing movie in the dataset (about $2.92 billion), followed by Avengers: Endgame and Avatar: The Way of Water.

2. Most of the top 10 movies are tagged Adventure, Action and Science Fiction. The exceptions are Titanic (Drama, Romance), Zootopia 2 and Inside Out 2 (Animation, Family) and Jurassic World, which is also Thriller.

Note: the results are sorted by revenue in descending order and limited to the top 10 movies. Genres are combined with GROUP_CONCAT so each movie appears once.

Recommendation: Adventure, Action and Science Fiction films dominate the highest-grossing list, so they are a strong choice for studios aiming at large box-office returns. This is based on only 10 movies, so it should be confirmed with the average revenue per genre in Q3.



## Average budget and revenue per genre

In [6]:
q3 = """SELECT g.genre_name, ROUND(AVG(m.budget),0) budgetAvrg, ROUND(AVG(m.revenue),0) revenueAvrg FROM movies m
JOIN  movie_genres mg ON m.movie_id = mg.movie_id
JOIN genres g ON mg.genre_id = g.genre_id
WHERE m.budget > 0 AND m.revenue > 0
GROUP BY g.genre_name
ORDER BY revenueAvrg DESC
"""

print(pd.read_sql(q3,engine))

         genre_name   budgetAvrg  revenueAvrg
0         Adventure  111393307.0  401451192.0
1         Animation   88282289.0  382103463.0
2            Family   88408767.0  357870775.0
3   Science Fiction   97864404.0  335689706.0
4            Action   94651416.0  309125778.0
5           Fantasy   89183665.0  307912995.0
6            Comedy   54652525.0  223039972.0
7             Music   42817021.0  205843825.0
8           Romance   37226946.0  177226878.0
9          Thriller   50844282.0  170049287.0
10              War   55774188.0  166091465.0
11            Crime   46559587.0  154320253.0
12            Drama   40050847.0  146720290.0
13          Mystery   38209954.0  145250653.0
14          Western   55748387.0  143540585.0
15          History   53532727.0  143006828.0
16           Horror   25757727.0  115506943.0
17         TV Movie    4200000.0       3746.0


### Insight

1. Adventure has the highest average revenue (about $401M), followed by Animation (about $382M) and Family (about $358M). Science Fiction and Action come next.

2. These high-revenue genres also have higher average budgets. Animation earns a similar revenue to Adventure with a lower average budget (about $88M vs $111M).

3. TV Movie has a very low average revenue (about $3,746), far below every other genre. The number of movies behind this average is not shown, so it should be interpreted with caution.

4. This supports the Q2 result, where most of the top-grossing films were Adventure, Action and Science Fiction. Q3 is stronger evidence, because it uses all movies with a known budget and revenue instead of only 10.

Note: movies with a budget of 0 or revenue of 0 were excluded, since 0 means the value is unknown. The results are grouped by genre and sorted by average revenue in descending order.

Recommendation: Studios aiming for high revenue can prioritise Adventure, Animation and Family films. Animation looks especially efficient, with high revenue at a lower budget. This compares revenue only, so profit and ROI should be checked in the Pandas analysis before making a final decision.



## Top 10 actors by number of movies 

In [7]:
q4 = """SELECT actor_name, COUNT(DISTINCT movie_id) movieCount FROM cast
GROUP BY person_id, actor_name
ORDER BY movieCount DESC
LIMIT 10
"""

print(pd.read_sql(q4,engine))

           actor_name  movieCount
0   Samuel L. Jackson          39
1           Brad Pitt          38
2      Robert De Niro          38
3         Johnny Depp          37
4           Tom Hanks          33
5  Scarlett Johansson          32
6        Willem Dafoe          32
7       Mark Wahlberg          32
8      Morgan Freeman          31
9          Tom Cruise          31


### Insight

1. Samuel L. Jackson has appeared in the most movies (39), followed by Brad Pitt and Robert De Niro, who are tied for second place with 38 movies each. Johnny Depp is next with 37.

2. The top 10 actors are close together, ranging from 31 to 39 movies, so no single actor is far ahead of the rest.

Note: the results are grouped by person_id and actor_name, so two different actors with the same name are not merged. COUNT(DISTINCT movie_id) is used so that an actor who played more than one character in the same movie is counted once for that movie. The list is sorted by movie count in descending order and limited to the top 10.

Recommendation: Actors with the most appearances are frequently cast, so they are a useful starting point for casting decisions. This counts appearances only, so it should be combined with revenue and rating data to judge their actual impact on a film's success.



## Directors with more than 3 movies 

In [8]:
q5 = """SELECT person_name AS director, COUNT(DISTINCT movie_id) AS movieCount FROM crew
WHERE job = 'Director'
GROUP BY person_id, person_name
HAVING COUNT(DISTINCT movie_id) > 3
ORDER BY movieCount DESC
"""

print(pd.read_sql(q5,engine))

              director  movieCount
0     Steven Spielberg          27
1           Tim Burton          19
2         Ridley Scott          19
3      Martin Scorsese          16
4      Robert Zemeckis          15
..                 ...         ...
212      Robert Eggers           4
213       Ryan Coogler           4
214    Andy Muschietti           4
215           Wes Ball           4
216  David F. Sandberg           4

[217 rows x 2 columns]


### Insight

1. 217 directors have directed more than 3 movies. Steven Spielberg leads with 27 movies, which is 8 more than Tim Burton and Ridley Scott, who are tied for second place with 19 movies each.
2. The list ends at 4 movies, where many directors are tied (for example Robert Eggers, Ryan Coogler, Andy Muschietti, Wes Ball and David F. Sandberg).
3. There are 1,123 unique directors in the dataset, so only about 19% have directed more than 3 movies. Most directors appear with only a few films. (SELECT COUNT(DISTINCT person_id) AS Count FROM crew WHERE job = 'Director')

Note: the data comes from the crew table, filtered to job = 'Director'. The results are grouped by person_id and person_name, so two different directors with the same name are not merged. HAVING COUNT(DISTINCT movie_id) > 3 keeps only directors with more than 3 movies, and the list is sorted by movie count in descending order.

Recommendation: Directors with many movies, such as Steven Spielberg, have a long track record that studios can review before hiring. Movie count alone does not show quality or box-office success, so it should be combined with revenue and rating data.





## Top 10 most frequent keywords

In [9]:
q6 = """SELECT keyword_name AS keyword, COUNT(DISTINCT movie_id) AS usageCount FROM movie_keywords
GROUP BY keyword_id, keyword_name
ORDER BY usageCount DESC
LIMIT 10
"""

print(pd.read_sql(q6,engine))

                  keyword  usageCount
0  based on novel or book         449
1                  sequel         416
2    duringcreditsstinger         296
3     aftercreditsstinger         242
4                  murder         176
5     based on true story         166
6           new york city         161
7                 villain         160
8            3d animation         155
9          based on comic         154


### Insight

1. "based on novel or book" is the most used keyword (449 movies), followed by "sequel" (416 movies). Together they stand well ahead of the rest of the top 10, where the 10th keyword ("based on comic") has 154 movies.

2. "duringcreditsstinger" (296) and "aftercreditsstinger" (242) are next. These describe post-credit scenes, so they are technical tags rather than story themes.

3. Other top keywords include "murder", "based on true story", "new york city", "villain" and "3d animation".

Note: the results count how many different movies use each keyword (COUNT(DISTINCT movie_id)). They are grouped by keyword_id and keyword_name, since keyword_id is unique for each keyword. The list is sorted by usage count in descending order and limited to the top 10.

Recommendation: Stories adapted from books or comics, and sequels, appear very often in the dataset, so existing source material is a common starting point for movies. Whether these films earn more than original stories is not shown here, so it should be checked against revenue and rating data.



## Budget above the overall average, revenue below the overall average 

In [10]:
q7 = """ SELECT title, budget, revenue FROM movies
WHERE budget > (SELECT AVG(budget) FROM movies WHERE budget > 0)
AND revenue < (SELECT AVG(revenue) FROM movies WHERE revenue > 0)
AND revenue > 0
ORDER BY budget DESC
"""

print(pd.read_sql(q7,engine))

                                      title       budget      revenue
0                               The Marvels  274800000.0  206136825.0
1                                   Red One  250000000.0  185700759.0
2    Wake Up Dead Man: A Knives Out Mystery  210000000.0    4000000.0
3                         Wonder Woman 1984  200000000.0  169601036.0
4                                      Luca  200000000.0   51074773.0
..                                      ...          ...          ...
268                               Rambo III   63000000.0  189015611.0
269                         The Equalizer 2   62000000.0  190400157.0
270                                Æon Flux   62000000.0   53321673.0
271                                 Shooter   61000000.0   95700000.0
272                                  2 Guns   61000000.0  131940411.0

[273 rows x 3 columns]


### Insight

1. 273 movies have a budget above the average budget and a known revenue below the average revenue.

2. The Marvels has the highest budget in this list (about $274.8M), but its revenue (about $206.1M) is lower than its budget. Red One, Wonder Woman 1984 and Luca show the same pattern. Luca stands out, with a $200M budget and about $51.1M revenue.

Note: movies with budget 0 or revenue 0 were excluded, since 0 means the value is unknown. Both averages are calculated using subqueries that also ignore zero values. The list is sorted by budget in descending order.

Recommendation: A high budget does not guarantee high revenue. Studios should compare expected revenue with the budget before committing to expensive productions. This data excludes marketing costs, so the true profit is not shown. A full profit and ROI analysis follows in the Pandas section.



## Actors who appeared in movies directed by Christopher Nolan 

In [11]:
q8 = """ SELECT DISTINCT ca.actor_name FROM cast AS ca
JOIN crew AS cr ON ca.movie_id = cr.movie_id
WHERE cr.person_name = 'Christopher Nolan'
AND cr.job = 'Director'
ORDER BY ca.actor_name
"""

print(pd.read_sql(q8,engine))

           actor_name
0       Aaron Eckhart
1           Al Pacino
2      Alon Aboutboul
3         Andy Serkis
4     Aneurin Barnard
..                ...
79          Tom Hardy
80      Tom Wilkinson
81       Topher Grace
82        Wes Bentley
83  Yuri Kolokolnikov

[84 rows x 1 columns]


### Insight

1. 84 different actors appeared in movies directed by Christopher Nolan. Each actor is listed once, even if they worked in more than one of his films.

Note: the cast table is joined with the crew table on movie_id. The filter job = 'Director' keeps only movies he directed, not movies where he was only a writer. SELECT DISTINCT removes repeated names, and the list is sorted alphabetically by actor name.

The cast table holds only the top 10 billed actors per movie, so the 84 actors are top-billed cast members, not every person who appeared in his films.

Recommendation: The same query works for any director by changing the name, so it can be used to find the actors a director often works with. It shows who has worked with them, but not how often.



## Genre with the highest average rating (genres with at least 20 movies)

In [12]:
q9 = """ SELECT g.genre_name, ROUND(AVG(m.vote_average),2) AS averageRating, COUNT(DISTINCT m.movie_id) AS movieCount
FROM genres g
JOIN movie_genres mg ON g.genre_id = mg.genre_id
JOIN movies m ON mg.movie_id = m.movie_id
GROUP BY g.genre_id, g.genre_name
HAVING COUNT(DISTINCT m.movie_id) >= 20
ORDER BY averageRating DESC
"""

print(pd.read_sql(q9,engine))

         genre_name  averageRating  movieCount
0               War           7.36          85
1           History           7.34         112
2             Music           7.32          49
3             Drama           7.25         937
4         Animation           7.20         246
5           Western           7.13          33
6           Romance           7.03         349
7             Crime           7.01         387
8            Family           6.94         344
9           Mystery           6.88         216
10         Thriller           6.86         673
11        Adventure           6.83         674
12          Fantasy           6.83         396
13           Comedy           6.81         762
14  Science Fiction           6.81         463
15           Action           6.73         808
16           Horror           6.62         339


### Insight

1. 17 of the 19 genres have at least 20 movies. The other 2 genres were removed by the HAVING condition.

2. War (7.36), History (7.34) and Music (7.32) have the highest average ratings, but they are very close, with differences of only about 0.02 between them. Drama (7.25) and Animation (7.20) are also close behind.

3. War has 85 movies and History has 112, while Drama has the most movies of all (937). Western has the fewest among the listed genres (33 movies), so its average rating is based on a smaller sample.

4. Horror has the lowest average rating (6.62).

Note: genres, movie_genres and movies are joined on their ID columns. Results are grouped by genre_id and genre_name, and HAVING COUNT(DISTINCT movie_id) >= 20 keeps only genres with at least 20 movies. A movie with several genres is counted once in each of its genres. Ratings are rounded to 2 decimals and the list is sorted by average rating in descending order.

Recommendation: War, History and Music films are rated highest by audiences, but the gap between genres is small, so genre alone does not decide a film's rating. For a more dependable rating signal, Drama is the strongest, because it is based on the most movies.

## Top 10 movies with the most keywords

In [13]:
q10 = """ SELECT m.title, COUNT(DISTINCT mk.keyword_id) AS keywordCount FROM movies m
JOIN movie_keywords mk ON m.movie_id = mk.movie_id
GROUP BY m.movie_id, m.title
ORDER BY keywordCount DESC
LIMIT 10
"""

print(pd.read_sql(q10,engine))

                                        title  keywordCount
0                                     Smile 2            98
1                                     Taken 3            70
2                                 Silent Hill            63
3                              Twelve Monkeys            57
4              War for the Planet of the Apes            54
5                                 The Shining            52
6              Dawn of the Planet of the Apes            51
7                    How to Train Your Dragon            50
8  How to Train Your Dragon: The Hidden World            49
9                       Pacific Rim: Uprising            49


### Insight

1. Smile 2 has the most keywords (98), far ahead of Taken 3 (70), which is 28 keywords lower. Silent Hill is third with 63.

2. From fifth place downward the counts are close together (54, 52, 51, 50, 49, 49). The 9th and 10th movies are tied at 49 keywords.

Note: movies and movie_keywords are joined on movie_id. Results are grouped by movie_id and title, and COUNT(DISTINCT keyword_id) counts the different keywords for each movie. The list is sorted by keyword count in descending order and limited to the top 10.

Recommendation: The number of keywords depends on how thoroughly a movie has been tagged, so it should not be used alone as a measure of a movie's theme or success. For recommendation systems that match movies by keywords, movies with many keywords give more detail to match on, while movies with few keywords may need extra tags.

### Overall recommendation: Adventure, Animation and Family genres bring the highest average revenue, while War, History and Music are rated highest. Studios should balance both, and confirm with profit and ROI in the Pandas analysis.” 
